In [ ]:
import pandas as pd

inputs = pd.read_csv('/Users/mx23826/Documents/ML_winter_school/BoFlex/SDL Case Studies/Inorganic Nanomaterials/Case2_MHP NCs/Dataset/inputs.csv')
outputs = pd.read_csv('/Users/mx23826/Documents/ML_winter_school/BoFlex/SDL Case Studies/Inorganic Nanomaterials/Case2_MHP NCs/Dataset/outputs.csv')

In [ ]:
inputs

In [ ]:
import corner

fig = corner.corner(inputs, labels=inputs.columns)

In [ ]:
# Check for correlations between features

In [ ]:
from sklearn.decomposition import PCA
from umap import UMAP

# reducer = PCA(n_components=2)
reducer = UMAP(n_components=2, random_state=42, metric='euclidean')

data = reducer.fit_transform(inputs)

import matplotlib.pyplot as plt

plt.scatter(data[:, 0], data[:, 1], )

In [ ]:
from sklearn.model_selection import train_test_split
import torch

total = pd.concat([inputs, outputs], axis=1)
train, test = train_test_split(total, test_size=0.1, random_state=42)

train_X=train.iloc[:, :inputs.shape[1]]
train_Y=train.iloc[:, -outputs.shape[1]:]
test_X=test.iloc[:, :inputs.shape[1]]
test_Y=test.iloc[:, -outputs.shape[1]:]

train_X = torch.tensor(train_X.values)
train_Y = torch.tensor(train_Y.values)
test_X = torch.tensor(test_X.values)
test_Y = torch.tensor(test_Y.values)

In [ ]:
# Following this tutorial for MOBO with botorch: https://jonathan-guerne.medium.com/multi-objective-bayesian-optimization-with-botorch-3c5cf348c63b
# And the GP_BO_Exercises notebook

from botorch.models import SingleTaskGP, ModelListGP
from gpytorch.mlls import ExactMarginalLogLikelihood, SumMarginalLogLikelihood
from botorch.fit import fit_gpytorch_mll
from botorch.acquisition import UpperConfidenceBound

models = []
for i in range(outputs.shape[1]):
    model = SingleTaskGP(train_X, train_Y[:, i].unsqueeze(1))
    models.append(model)
    
gp = ModelListGP(*models)
mll = SumMarginalLogLikelihood(gp.likelihood, gp)
fit_gpytorch_mll(ExactMarginalLogLikelihood(model.likelihood,model))
acq=UpperConfidenceBound(model=model,beta=2.0**2)